# Lab 18-04 · Buffers, Bus Contention and a Lock-Free Ring

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import dmasim as dm
import fwrun

## Part 1 · Block size

*Predict* the smallest safe block for Example 18-04.2 with two and three buffers.

In [2]:
fs, a, b = 1e6, 20e-6, 0.6e-6
print("bounds:", dm.pingpong_min_block(fs, a, b, 25e-6, 2), dm.pingpong_min_block(fs, a, b, 25e-6, 3))
for n, k in ((64, 2), (112, 2), (128, 2), (48, 3), (64, 3)):
    ov, worst = dm.simulate_pingpong(fs, n, a, b, n_blocks=20000, n_buffers=k, service_latency=5e-6, jitter=20e-6, seed=1)
    print(f"N = {n:3d}, {k} buffers: {ov:5d} overruns in 20000 blocks, worst latency {worst * 1e6:.1f} us")

bounds: 112.5 50.0
N =  64, 2 buffers: 19901 overruns in 20000 blocks, worst latency 147.4 us
N = 112, 2 buffers:   191 overruns in 20000 blocks, worst latency 224.2 us
N = 128, 2 buffers:     0 overruns in 20000 blocks, worst latency 249.8 us
N =  48, 3 buffers: 19970 overruns in 20000 blocks, worst latency 16119.2 us
N =  64, 3 buffers:     0 overruns in 20000 blocks, worst latency 147.4 us


## Part 2 · Latency against memory

In [3]:
L = np.linspace(0, 60e-6, 61)
for k in (2, 3):
    n = np.array([2 ** np.ceil(np.log2(dm.pingpong_min_block(fs, a, b, x, k))) for x in L])
    plt.plot(L * 1e6, (n / fs + L + a + b * n) * 1e6, label=f"{k} buffers")
plt.xlabel("worst service latency (us)"); plt.ylabel("worst sample latency (us)"); plt.legend(); plt.show()

## Part 3 · Contention

*Predict* the stall fraction for p = 0.3 and DMA periods of 25, 10 and 4 cycles.

In [4]:
for per in (25, 10, 4):
    r1 = dm.bus_sim(1_000_000, 0.3, per)
    r2 = dm.bus_sim(1_000_000, 0.3, per, "round_robin")
    print(f"period {per:2d}: estimate {100 * dm.stall_estimate(0.3, 1 / per):.2f}%, DMA-first {100 * r1['stall_fraction']:.2f}%, round-robin {100 * r2['stall_fraction']:.2f}%")

period 25: estimate 1.20%, DMA-first 1.21%, round-robin 1.21%


period 10: estimate 3.00%, DMA-first 3.02%, round-robin 3.02%


period  4: estimate 7.50%, DMA-first 7.52%, round-robin 7.52%


## Part 4 · The ring buffer on the core

*Predict* what the firmware checks when the consumer stops.

In [5]:
for opt in ("-O2", "-O0"):
    status, out, _ = fwrun.build_and_run(Path(tempfile.mkdtemp(prefix="lab1804_")), opt)
    print(opt, "exit status", status, "|", out.strip())

-O2 exit status 0 | TEST PASSED


-O0 exit status 0 | TEST PASSED


## Part 5 · Cache lines

In [6]:
for addr, n in ((0x20000008, 1000), (0x20010014, 600), (0x20000000, 1024)):
    print(f"0x{addr:08X}, {n} bytes: {dm.cache_lines(addr, n)} lines; aligned and padded size {-(-n // 32) * 32} bytes")

0x20000008, 1000 bytes: 32 lines; aligned and padded size 1024 bytes
0x20010014, 600 bytes: 20 lines; aligned and padded size 608 bytes
0x20000000, 1024 bytes: 32 lines; aligned and padded size 1024 bytes


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).